# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/M-Abdullah-Nazeer/Flyrank-Ml-Pipeline/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
!pip -q install duckdb
import duckdb, os
import pandas as pd, numpy as np
from google.colab import userdata
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupShuffleSplit

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")
rel = "hf://datasets/FlyRank/internship-warehouse"

In [2]:
def build_month(m):
    con.sql(f"""CREATE OR REPLACE TABLE dm AS
      SELECT * FROM read_parquet('{rel}/fact_content_daily_performance/month={m}/*.parquet')""")
    df = con.sql("""
    WITH past AS (
      SELECT client_hash_id, content_hash_id,
             SUM(gsc_impressions) AS impressions, SUM(gsc_clicks) AS clicks,
             SUM(gsc_avg_position * gsc_impressions) / NULLIF(SUM(gsc_impressions), 0) AS position,
             COUNT(*) AS days_with_data, MAX(day(report_date)) AS last_day_used,
             SUM(CASE WHEN day(report_date) BETWEEN 8 AND 15 THEN gsc_impressions END) AS impr_late,
             SUM(CASE WHEN day(report_date) BETWEEN 1 AND 7 THEN gsc_impressions END) AS impr_early
      FROM dm WHERE gsc_data_available IS TRUE AND gsc_impressions > 0 AND day(report_date) <= 15
      GROUP BY 1, 2),
    fut AS (
      SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS f_impr, SUM(gsc_clicks) AS f_clicks
      FROM dm WHERE gsc_data_available IS TRUE AND day(report_date) > 15
      GROUP BY 1, 2)
    SELECT p.*, f.f_impr, f.f_clicks FROM past p JOIN fut f USING (client_hash_id, content_hash_id)
    """).df()
    df["month"] = m
    return df

MONTHS = ["2026-02", "2026-03", "2026-04"]       # same months as Week 5; the sealed 2026-06 is not used
assert "2026-06" not in MONTHS
raw = pd.concat([build_month(m) for m in MONTHS], ignore_index=True)
pages = raw[(raw["impressions"] >= 100) & (raw["position"] <= 20) & (raw["f_impr"] >= 100)].reset_index(drop=True)
print(pages.shape, pages["month"].value_counts().to_dict())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(165928, 12) {'2026-04': 59986, '2026-03': 58123, '2026-02': 47819}


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**Source:** FlyRank Data Report, "The State of AI-Driven SEO. In Numbers." (April 2026). I read it as a self-learner. The paper states its own limits clearly: it calls itself a pattern study and not proof of cause, defines its reporting windows, and flags a very small sample in one freshness bucket. I'm asking these questions to learn how to disclose the same things in my own work.

**Finding 1 ("Which Pages Will Grow?", ML appendix):** A model trained on 96.6K pages that were clearly growing or declining reaches about 90% accuracy on unseen pages from the same brands and about 75% on brands it never saw, with days visible and days since update as the top predictors.
**Question about the label:** The report defines trend direction by comparing the last 30 days with the 30 days before (up or down by more than 10%). The strongest predictor, days visible, is counted over the last 90 complete days. Do the features stop before the 30-day window that defines the label, or do they overlap with it? Knowing the exact cutoff would help me document mine.
**Question about validation:** Holding out whole brands is a strong design, and the 64%–85% range across splits is useful disclosure. To read 75% and 90%, I would like to see the share of growing pages in the test data and a simple baseline (always guessing "growing"). Finding 1 reports about 74.5K growing and 46.2K declining pages, so such a guess could already score in the 60s. It would also help to know that pages with stable trends (inside the ±10% band) were left out, so the accuracy describes only clearly moving pages.
**Why I ask:** The paper gives per-split ranges and a calibration check, which is more than many reports do. Both questions are about learning where a baseline and a feature cutoff belong in a results table.

**Finding 2 ("Refreshing Pages Actually Works" / Freshness Multiplier):** Among pages older than a year, those refreshed in the last 30 days show about 1.6× the health score and about 52× the impressions of old pages last updated 181–360 days ago, with significant lift in 7 of 9 strata on held-out data.
**Question about the label:** Finding 8 explains that an "update" is detected through internal workflow logic and algorithmic signals, and may be a title edit, a body edit, a link change or a layout change. Which of these was counted as a refresh in this comparison, and is the update date always earlier than the impressions window it is compared against? If pages with rising impressions are more likely to be refreshed, the arrow could run the other way.
**Question about validation:** This reads as a comparison between two groups of pages (refreshed vs stale), not the same pages before and after. How were the refreshed pages chosen, and were pages matched on age, starting impressions or brand? The Limitations section says the study finds patterns and not proof of cause, which I appreciate. The section title ("Refreshing Pages Actually Works") reads more causally than that wording, and the Method section says no p-values or confidence intervals are reported, while Part IV reports significance tests and confidence intervals. I would like to understand which standard applies where.
**Why I ask:** The paper already separates a headline from its caveats. I'm practising writing the headline at the same strength as the evidence, in my own results.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

**Before:** a random row split, where pages from the same client and the same page in different months can sit on both sides. This is the easy version and is likely to look better than reality.
**After:** (B) clients held out, (C) time-aware: train on Feb+Mar, test on April (my Week-5 setup), (D) strictest: April pages of clients the model never saw in training. For every split, the expected-CTR curve is computed from the training rows only, and the Week-4 baseline is scored on the same test rows.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
FEATURES = ["log_impr", "position", "ctr", "ctr_gap_ratio", "trend_ratio", "days_with_data"]
bins, labs = [0, 3, 6, 10, 20], ["1-3", "4-6", "7-10", "11-20"]

def prep(train, test):
    tb = pd.cut(train["position"], bins, labels=labs)
    expected = (train.groupby(tb, observed=True)["clicks"].sum() /
                train.groupby(tb, observed=True)["impressions"].sum()).to_dict()   # training rows only
    outs = []
    for df in (train, test):
        df = df.copy()
        df["expected_ctr"] = pd.cut(df["position"], bins, labels=labs).map(expected).astype(float)
        df["ctr"] = df["clicks"] / df["impressions"]
        df["log_impr"] = np.log10(df["impressions"])
        df["ctr_gap_ratio"] = (df["ctr"] / df["expected_ctr"]).clip(upper=10)
        df["trend_ratio"] = (df["impr_late"] / df["impr_early"]).replace([np.inf, -np.inf], np.nan).clip(upper=10)
        df["missed_clicks"] = df["impressions"] * (df["expected_ctr"] - df["ctr"])      # Week-4 score
        df["label"] = (df["f_clicks"] < 0.5 * df["f_impr"] * df["expected_ctr"]).astype(int)
        outs.append(df)
    return outs

def prec_at_k(score, y, k):
    idx = np.argsort(-np.asarray(score))[:k]
    return float(np.asarray(y)[idx].mean())

def evaluate(train, test, name):
    tr, te = prep(train, test)
    model = HistGradientBoostingClassifier(random_state=0).fit(tr[FEATURES], tr["label"])
    te["model_score"] = model.predict_proba(te[FEATURES])[:, 1]
    y = te["label"].values
    return {"split": name, "n_test": len(te), "base_rate": round(y.mean(), 3),
            "AUC baseline": round(roc_auc_score(y, te["missed_clicks"]), 3),
            "AUC model": round(roc_auc_score(y, te["model_score"]), 3),
            "p@500 baseline": round(prec_at_k(te["missed_clicks"], y, 500), 3),
            "p@500 model": round(prec_at_k(te["model_score"], y, 500), 3)}, tr, te, model

rows = []
# A: BEFORE, random rows
rng = np.random.RandomState(0); perm = rng.permutation(len(pages)); cut = int(0.75 * len(pages))
r, *_ = evaluate(pages.iloc[perm[:cut]], pages.iloc[perm[cut:]], "A. random rows (before)"); rows.append(r)
# B: clients held out
tr_i, te_i = next(GroupShuffleSplit(test_size=0.25, random_state=0).split(pages, groups=pages["client_hash_id"]))
r, *_ = evaluate(pages.iloc[tr_i], pages.iloc[te_i], "B. clients held out"); rows.append(r)
# C: time-aware (Week-5 setup)
r, tr_c, te_c, model_c = evaluate(pages[pages["month"] != "2026-04"], pages[pages["month"] == "2026-04"],
                                  "C. time-aware (train Feb+Mar, test Apr)"); rows.append(r)
# D: later month AND unseen clients
rng = np.random.RandomState(1); clients = pages["client_hash_id"].unique()
held = set(rng.choice(clients, size=max(1, int(0.25 * len(clients))), replace=False))
r, *_ = evaluate(pages[(pages["month"] != "2026-04") & ~pages["client_hash_id"].isin(held)],
                 pages[(pages["month"] == "2026-04") & pages["client_hash_id"].isin(held)],
                 "D. later month + unseen clients"); rows.append(r)

before_after = pd.DataFrame(rows)
display(before_after)
os.makedirs("work/outputs", exist_ok=True)
before_after.to_csv("work/outputs/w06_before_after.csv", index=False)

,split,n_test,base_rate,AUC baseline,AUC model,p@500 baseline,p@500 model
0,A. random rows (before),41482,0.496,0.695,0.787,0.764,0.928
1,B. clients held out,69221,0.494,0.697,0.792,0.754,0.932
2,"C. time-aware (train Feb+Mar, test Apr)",59986,0.497,0.680,0.788,0.786,0.924
3,D. later month + unseen clients,7657,0.537,0.679,0.769,0.778,0.898


**Before/after (gradient boosting vs the Week-4 baseline, same test rows in each split):**

| Split | n test | Base rate | AUC baseline → model | p@500 baseline → model |
|---|---|---|---|---|
| A. random rows (before) | 41,482 | 0.496 | 0.695 → 0.787 | 0.764 → 0.928 |
| B. clients held out | 69,221 | 0.494 | 0.697 → 0.792 | 0.754 → 0.932 |
| C. time-aware (train Feb+Mar, test Apr) | 59,986 | 0.497 | 0.680 → 0.788 | 0.786 → 0.924 |
| D. later month + unseen clients | 7,657 | 0.537 | 0.679 → 0.769 | 0.778 → 0.898 |

**What I observed:** the honest splits did not make the model look much worse than the random split. Model AUC was 0.787 under random rows and 0.788 / 0.792 under time-aware and client-held-out splits. Only the strictest split (D: later month and clients the model never saw) is lower, at 0.769, about 0.02 below the random split. In every split the model stays ahead of the Week-4 baseline: in D the lead is about 0.09 AUC and 12 points of precision@500 (0.898 vs 0.778). Split D has only 7,657 test pages from one random draw of held-out clients, so I treat its numbers as directional. My Week-5 client check (AUC 0.746) used a different, smaller set (training months only), so it is not directly comparable to B or D, and I use B and D as the cleaner numbers.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*



In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# 1) Every feature comes from days 1-15 only
print("latest day-of-month used by features:", int(pages["last_day_used"].max()))
assert pages["last_day_used"].max() <= 15
print("features:", FEATURES)
print("label uses days 16+ (f_clicks, f_impr); expected CTR comes from training rows only")

# 2) Single-feature smell test: a feature that alone gives AUC near 1.0 is suspicious
for f in FEATURES:
    s = te_c[f].fillna(te_c[f].median())
    a = roc_auc_score(te_c["label"], s)
    print(f"{f:16s} single-feature AUC = {max(a, 1 - a):.3f}")

# 3) Page overlap: how many April pages also appear in the training months?
key = lambda d: d["client_hash_id"] + "|" + d["content_hash_id"]
overlap = key(te_c).isin(set(key(tr_c)))
print("share of test pages also seen in training months:", round(overlap.mean(), 3))
print("AUC on test pages never seen in training:",
      round(roc_auc_score(te_c.loc[~overlap, "label"], te_c.loc[~overlap, "model_score"]), 3))

# 4) The trap, again: a column from the label window makes the score jump
leaky = FEATURES + ["f_clicks"]
m_leak = HistGradientBoostingClassifier(random_state=0).fit(tr_c[leaky], tr_c["label"])
print("AUC with deliberate leak (f_clicks):", round(roc_auc_score(te_c["label"], m_leak.predict_proba(te_c[leaky])[:, 1]), 3),
      "| honest:", round(roc_auc_score(te_c["label"], te_c["model_score"]), 3))

latest day-of-month used by features: 15
features: ['log_impr', 'position', 'ctr', 'ctr_gap_ratio', 'trend_ratio', 'days_with_data']
label uses days 16+ (f_clicks, f_impr); expected CTR comes from training rows only
log_impr         single-feature AUC = 0.619
position         single-feature AUC = 0.581
ctr              single-feature AUC = 0.748
ctr_gap_ratio    single-feature AUC = 0.745
trend_ratio      single-feature AUC = 0.531
days_with_data   single-feature AUC = 0.512
share of test pages also seen in training months: 0.797
AUC on test pages never seen in training: 0.769
AUC with deliberate leak (f_clicks): 0.985 | honest: 0.788


**Leakage audit:** all six features are computed from days 1-15 (checked above: the latest day used is 15). The label needs days 16 onward, which no feature uses, and the expected-CTR curve comes from training rows only. No single feature alone predicts the label near-perfectly: the strongest are `ctr` (AUC 0.748) and `ctr_gap_ratio` (0.745), and the others are 0.51-0.62, so there is no sign of a hidden copy of the label. About 79.7% of April pages also appeared in the training months. On pages never seen in training, AUC is 0.769 against 0.788 overall, so seeing the same page earlier helps by roughly 0.02 AUC but is not what drives the result. The deliberate leak (`f_clicks`) raised AUC to 0.985, which confirms this audit would show a leak if there were one. I did not use product flags, health scores or recommended actions as inputs.

In [6]:
cols = ["position", "impressions", "ctr", "expected_ctr", "model_score"]
fp = te_c[te_c["label"] == 0].sort_values("model_score", ascending=False).head(10).copy()
fp["future_ctr"] = fp["f_clicks"] / fp["f_impr"]
print("FALSE POSITIVES: ranked high, but did not under-capture later")
display(fp[cols + ["future_ctr"]].round(4))

fn = te_c[te_c["label"] == 1].sort_values("model_score").head(5).copy()
fn["future_ctr"] = fn["f_clicks"] / fn["f_impr"]
print("FALSE NEGATIVES: ranked low, but did under-capture later")
display(fn[cols + ["future_ctr"]].round(4))

FALSE POSITIVES: ranked high, but did not under-capture later


,position,impressions,ctr,expected_ctr,model_score,future_ctr
122214,1.8288,7336.0,0.0001,0.0042,0.9542,0.0044
128574,1.6961,7051.0,0.0004,0.0042,0.9542,0.0117
120914,0.5925,4106.0,0.0002,0.0042,0.9535,0.0022
158409,6.0355,28631.0,0.0002,0.0030,0.9508,0.0016
150873,1.7041,7868.0,0.0000,0.0042,0.9498,0.0033
120657,6.7105,10032.0,0.0004,0.0030,0.9480,0.0017
155523,2.1833,2951.0,0.0003,0.0042,0.9474,0.0027
114046,9.6670,7112.0,0.0003,0.0030,0.9444,0.0017
111194,3.9816,7355.0,0.0004,0.0036,0.9438,0.0019
158198,3.7470,4399.0,0.0002,0.0036,0.9426,0.0019


FALSE NEGATIVES: ranked low, but did under-capture later


,position,impressions,ctr,expected_ctr,model_score,future_ctr
145291,2.3263,4646.0,0.0359,0.0042,0.0079,0.0000
139114,3.4712,7186.0,0.0074,0.0036,0.0149,0.0017
141878,2.4157,1975.0,0.0177,0.0042,0.0195,0.0000
141494,4.0678,1416.0,0.0092,0.0036,0.0214,0.0000
147333,10.0210,7805.0,0.0060,0.0032,0.0256,0.0012


**False positives (ranked high, did not under-capture):** all ten had model scores around 0.94-0.95, very low CTR in days 1-15 (0.0000-0.0004) against an expected 0.3-0.4%, and large volume (about 3,000-29,000 impressions). Compared with the label's cutoff (half the expected CTR), most were near misses: their future CTR of about 0.16-0.27% sat just above the line, and two recovered fully (0.44% and 1.17%). So many "wrong" picks are borderline pages, not healthy ones. The 50% cutoff is a choice I made, and it makes some picks look wrong that a reviewer would still call worth checking.
**False negatives (ranked low, did under-capture):** all five had a CTR above expected in days 1-15 (for example 3.6% against 0.4% expected) and then fell to near zero clicks later in the month (future CTR 0.00-0.17%). Nothing in the first half warns about a sudden later drop, such as a rank loss or a tracking change, so the model cannot be expected to catch these with these features.
**What a reviewer should check before acting on a pick:** whether the page is a near miss or a clear case, whether a search feature or branded query explains the low CTR, and whether position changed during the window.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Original (too strong):** "My model finds pages that will keep losing clicks, so these pages need their titles rewritten."
**Step 1, remove causation:** "The model ranks pages that are likely to keep getting fewer clicks than expected for their position."
**Step 2, add the evidence and scope:** "On one held-out month (2026-04), 92.4% of the top 500 pages ranked by gradient boosting under-captured clicks, against 78.6% for the Week-4 baseline and a 49.7% base rate."
**Step 3, add the validation limit:** "With a later month and clients the model never saw (split D, n = 7,657 pages), the top-500 figure is 89.8% against 77.8% for the baseline and a 53.7% base rate, so the lead stays at about 12 points, though it comes from a single draw of held-out clients."
**Final version I can stand behind:** "Observed on three months of Search Console data, a gradient-boosting ranker is a directional decision-support tool for deciding which pages to review first. It stayed ahead of a simple rule under every split I tried, including unseen clients in a later month. It does not show why a page under-captures or that a title rewrite would raise clicks."

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.